<a href="https://colab.research.google.com/github/Leonanda1013/DataLoverz/blob/main/Competition/Playground/s6e5/2_XGBoost_F1_PitStop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XGBoost – F1 Pit Stop Prediction
**Dataset:** Kaggle Playground Series S6E5
**Target:** `PitNextLap` (binary classification)

## Alur Notebook
1. Konfigurasi & Imports
2. Load Data
3. Preprocessing
4. Baseline Model
5. Hyperparameter Tuning (Optuna)
6. Evaluasi & Ekspor Submission

## 1. Konfigurasi

In [ ]:
# ── Ubah nilai di blok ini sesuai kebutuhan ──────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/DATASET/playground-series-s6e5'
TRAIN_PATH   = f'{DRIVE_BASE}/train.csv'
TEST_PATH    = f'{DRIVE_BASE}/test.csv'

TARGET_COL   = 'PitNextLap'
ID_COL       = 'id'

TEST_SIZE    = 0.2
RANDOM_STATE = 10

# Mapping manual untuk kolom Compound (kategori ban)
COMPOUND_MAPPING = {
    'WET': 0, 'SOFT': 1, 'MEDIUM': 2, 'INTERMEDIATE': 3, 'HARD': 4
}

# Kolom kategorikal yang di-encode dengan LabelEncoder
LABEL_ENCODE_COLS = ['Driver', 'Race']

# Parameter baseline XGBoost
BASELINE_PARAMS = {
    'n_estimators'    : 200,
    'max_depth'       : 6,
    'learning_rate'   : 0.1,
    'subsample'       : 0.8,
    'colsample_bytree': 0.8,
    'random_state'    : RANDOM_STATE,
}

# Konfigurasi Optuna
N_TRIALS = 50
CV_FOLDS = 5
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
!pip install optuna -q

import pandas as pd
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from google.colab import drive
drive.mount('/content/drive')

## 3. Load Data

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f'Train : {df_train.shape}')
print(f'Test  : {df_test.shape}')
df_train.head()

## 4. Preprocessing

In [ ]:
def preprocess(train: pd.DataFrame, test: pd.DataFrame) -> tuple:
    """Encode kolom kategorikal dan set index. Return (train, test) yang sudah bersih."""
    train = train.copy()
    test  = test.copy()

    # Encode kolom Compound dengan mapping yang sudah ditentukan
    train['Compound'] = train['Compound'].map(COMPOUND_MAPPING)
    test['Compound']  = test['Compound'].map(COMPOUND_MAPPING)

    # Encode kolom kategorikal lain dengan LabelEncoder
    for col in LABEL_ENCODE_COLS:
        le = LabelEncoder().fit(pd.concat([train[col], test[col]]))
        train[col] = le.transform(train[col])
        test[col]  = le.transform(test[col])

    # Set id sebagai index
    train.set_index(ID_COL, inplace=True)
    test.set_index(ID_COL, inplace=True)

    return train, test


df_train, df_test = preprocess(df_train, df_test)

X = df_train.drop(columns=[TARGET_COL])
y = df_train[TARGET_COL]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}')
df_train.info()

## 5. Baseline Model

In [ ]:
baseline = xgb.XGBClassifier(**BASELINE_PARAMS)
baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_val)
print(f'Accuracy  : {accuracy_score(y_val, y_pred):.4f}\n')
print('Confusion Matrix:')
print(confusion_matrix(y_val, y_pred))
print('\nClassification Report:')
print(classification_report(y_val, y_pred))

## 6. Hyperparameter Tuning (Optuna)
Cari kombinasi parameter terbaik menggunakan cross-validation.

In [ ]:
def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 500),
        'max_depth'       : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state'    : RANDOM_STATE,
    }
    model = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=CV_FOLDS, scoring='accuracy')
    return scores.mean()


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS)

print(f'Best CV Accuracy : {study.best_value:.4f}')
print(f'Best Params      : {study.best_params}')

## 7. Evaluasi & Ekspor Submission

In [ ]:
# Train ulang dengan parameter terbaik pada seluruh data training
best_model = xgb.XGBClassifier(**study.best_params, random_state=RANDOM_STATE)
best_model.fit(X, y)

# Evaluasi pada validation set
y_pred_best = best_model.predict(X_val)
print(f'Tuned Accuracy : {accuracy_score(y_val, y_pred_best):.4f}')
print(classification_report(y_val, y_pred_best))

# Buat submission
submission = pd.DataFrame({
    ID_COL    : df_test.index,
    TARGET_COL: best_model.predict(df_test),
})
submission.to_csv('submission.csv', index=False)
print('✓ submission.csv tersimpan.')
submission.head()